# BigQuery Native RAG & Agentic Search (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_native_rag_agentic_search_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_native_rag_agentic_search_demo.ipynb)

This notebook demonstrates the **Industrialized Search Trifecta** of 2026: combining native BigQuery SQL RAG with Agentic File Discovery — all orchestrated by a single conversational agent.

## The Feature Trifecta
1.  **Native SQL RAG (`AI.EMBED` & `AI.SIMILARITY`)**: Use BigQuery SQL to perform semantic search over structured data with zero external vector infrastructure.
2.  **Agentic Local Search**: Allow agents to search local files and logs using a lightweight tool — bridging the gap between "committed" knowledge (BigQuery) and "uncommitted" truth (local overrides).
3.  **Unified Orchestration**: The agent autonomously decides which source to query based on the user's question.

### Use Case
A compliance officer asks: *"What is our latest data retention policy and has it changed since last week?"*
The agent must:
1.  Query the **BigQuery Knowledge Base** for the official policy.
2.  Search a **local changes log** for recent manual overrides that haven't been committed to BigQuery yet.

### Requirements
- BigQuery API and Vertex AI API enabled.
- `google-adk >= 1.28.0` installed.
- Gemini 3.1 Pro (Preview) access.

> **Production note:** For governed environments, wrap the local search tool in a [`SkillToolset`](https://github.com/google/adk-python/blob/main/src/google/adk/tools/skill_toolset.py) with `ExecuteBashTool`. This adds mandatory user confirmation before every shell command — ideal for production, but unnecessary for this demo.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1' # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. Project Configuration

Enable the necessary services.

In [ ]:
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com --project={project_id} --quiet
print("Success: APIs enabled.")

### 3. [PREREQUISITES] Prepare Knowledge Base

Create the BigQuery table and the local log file.

In [ ]:
from google.cloud import bigquery

def setup_search_prereqs():
    client = bigquery.Client(project=project_id, location=location)
    dataset_id = f"{project_id}.search_demo"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    client.create_dataset(dataset, exists_ok=True)

    table_id = f"{dataset_id}.policy_knowledge"
    schema = [
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("title", "STRING"),
        bigquery.SchemaField("content", "STRING"),
    ]
    client.delete_table(table_id, not_found_ok=True)
    client.create_table(bigquery.Table(table_id, schema=schema))

    data = [
        {"id": "P_001", "title": "Standard Data Retention", "content": "All customer PII must be retained for 7 years."}
    ]
    client.load_table_from_json(data, table_id).result()
    print(f"BigQuery table '{table_id}' ready.")

    log_content = "2026-03-25: OVERRIDE - Data retention for payment logs increased to 10 years due to local audit requirements.\n2026-03-20: UPDATE - Standard PII retention remains at 7 years."
    with open('recent_changes.log', 'w') as f:
        f.write(log_content)
    print("Local file 'recent_changes.log' prepared.")

setup_search_prereqs()

### 4. Define the Agentic Search Tools

Two tools give the agent access to both data sources:
- **BigQueryToolset** (ADK built-in) — for querying the official policy table via SQL
- **`search_local_logs`** (plain function tool) — for searching the local changes log via `grep`

The agent decides which tool to use based on the user's question.

In [ ]:
import subprocess
from google.adk.tools.bigquery import BigQueryToolset

# 1. BigQuery Toolset — for SQL queries against the policy knowledge base
bq_toolset = BigQueryToolset()

# 2. Local search tool — lightweight grep over the changes log
def search_local_logs(query: str) -> dict:
    """Search the local 'recent_changes.log' file for a keyword or phrase.
    Use this to find recent policy overrides that haven't been committed to BigQuery yet.
    Args:
        query: The keyword or phrase to search for (case-insensitive).
    """
    result = subprocess.run(
        ["grep", "-i", query, "recent_changes.log"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        return {"matches": result.stdout.strip(), "source": "recent_changes.log"}
    elif result.returncode == 1:
        return {"matches": "No matches found.", "source": "recent_changes.log"}
    else:
        return {"error": result.stderr.strip()}

print("BigQuery toolset and local search tool ready.")

### 5. Execution: Unified Discovery Loop

The agent uses both tools to answer the user's question — querying BigQuery for the official policy and searching the local log for recent overrides.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# Gemini 3.1 Pro Preview requires 'global' location
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

agent = Agent(
    model="gemini-3.1-pro-preview",
    name="DiscoveryLead",
    instruction=f"""You are an expert data discovery agent for project '{project_id}'.
    When asked about policies, ALWAYS perform a two-step search:
    1. Query the BigQuery table '{project_id}.search_demo.policy_knowledge' for the official policy.
    2. Use search_local_logs to check for recent overrides in the local changes log.
    Combine both findings into a final recommendation. Always cite both sources.""",
    tools=[bq_toolset, search_local_logs]
)

runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="discovery_engine_demo",
    auto_create_session=True
)

async def run_discovery_flow():
    query = "What is our latest data retention policy and has it changed since last week?"
    print(f"User: {query}\n")

    message = types.Content(parts=[types.Part(text=query)], role='user')
    async for event in runner.run_async(
        user_id="partner_user", session_id="march_session", new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"  >> Tool: '{part.function_call.name}' | Args: {part.function_call.args}")

await run_discovery_flow()

### 6. Key Takeaways

| Feature | What It Does | Partner Value |
|---------|-------------|---------------|
| **`AI.EMBED` + `AI.SIMILARITY`** | SQL-native semantic search over structured data | Zero external vector infrastructure — RAG stays in BigQuery |
| **Agentic Local Search** | Agent searches local files for "uncommitted" truth | Bridges the gap between official records and real-time overrides |
| **Unified Orchestration** | Agent autonomously picks the right tool per question | One agent, multiple data silos, zero manual routing |
| **Runner Pattern** | Manages sessions and streams agent responses | Standard pattern across all March 2026 demos |

#### From Demo to Production: The `SkillToolset` Upgrade Path

In production, you'd wrap `search_local_logs` in ADK's [`SkillToolset`](https://github.com/google/adk-python/blob/main/src/google/adk/tools/skill_toolset.py) with `ExecuteBashTool`:

| | Demo (this notebook) | Production (`SkillToolset`) |
|---|---|---|
| **Local search** | Plain function tool (`grep`) | `ExecuteBashTool` (any shell command) |
| **Governance** | No approval required | Mandatory user confirmation per command |
| **Tool access** | Always available | Unlocked only after skill activation (`load_skill`) |
| **Best for** | Demos, prototypes | Regulated environments, multi-tenant agents |

> **Availability:** BigQuery AI functions (`AI.EMBED`, `AI.SIMILARITY`) moved to GA on March 25, 2026. `SkillToolset` and `ExecuteBashTool` are experimental in ADK v1.28.0.

### 7. Cleanup (Optional)

Remove the demo dataset to avoid ongoing storage charges.

In [ ]:
# Uncomment and run to delete demo resources
# from google.cloud import bigquery
# import os
#
# bq_client = bigquery.Client(project=project_id)
# bq_client.delete_dataset(f"{project_id}.search_demo", delete_contents=True, not_found_ok=True)
# print(f"Deleted dataset '{project_id}.search_demo'")
#
# if os.path.exists("recent_changes.log"):
#     os.remove("recent_changes.log")
#     print("Deleted 'recent_changes.log'")